# Step 0：A公司煤炭数据治理项目 — 背景与数据说明

## 1. 项目背景

A公司是一家大型煤炭能源集团，下辖 **5座矿井**，年产原煤约 **3000万吨**。集团总部设有安全生产、调度指挥、煤质管控、财务结算等多个业务中心，各中心依赖信息系统进行日常决策。

### 1.1 数据治理驱动力

| 驱动力 | 具体表现 |
|--------|----------|
| **安全生产合规** | 瓦斯、一氧化碳等监测数据需保存≥2年，异常告警须可追溯 |
| **经营分析需求** | 产销一体化分析需要打通ERP订单与PI生产实绩 |
| **数据孤岛严重** | SAP-ERP、PI-System、SCADA、LIMS、OA五个系统相互独立，无统一数据标准 |
| **质量问题频发** | 各系统数据质量问题频发（空值、重复、格式不一致），影响分析准确性 |
| **监管报送压力** | 煤质数据、产量数据需定期向能源局、应急管理部报送 |

---

## 2. 五大异构系统说明

本项目模拟了 A公司真实部署的 5 个异构信息系统，它们各自独立运行、数据标准不统一，构成了典型的「数据孤岛」场景。

### 2.1 SAP-ERP（Oracle关系型）

企业级管理软件，覆盖采购、生产、销售、库存、财务等核心业务流程。在本项目中管理：

- **VBAK** — 销售订单抬头表（每行一条订单）
- **VBAP** — 销售订单行项目表（一条订单有多行物料）
- **KNA1** — 客户主数据表

### 2.2 PI-System（TimescaleDB时序）

工业场景常用的时序数据 historian 系统，以标签（Tag）为单位连续采集传感器数据。覆盖：

- **WAGAS** — 瓦斯浓度（%）
- **TEMP** — 温度（℃）
- **CO** — 一氧化碳（ppm）
- **CO2** — 二氧化碳（ppm）
- **PRESS** — 压力（kPa）
- **FAN_SPEED** — 风机转速（RPM）

### 2.3 LIMS（SQL Server）

实验室信息管理系统，管理煤质化验数据：采样批次、灰分、挥发分、发热量、硫分等指标。

### 2.4 OA（流程引擎）

办公自动化系统，处理合同审批、付款申请、会议纪要等行政审批流程。

### 2.5 SCADA（工控系统/Kafka）

数据采集与监控系统，负责采集皮带机、排水泵、提升机等关键设备的实时状态监控，数据经 Kafka 实时流进入数据湖。

---

## 3. 数据孤岛问题

五个系统使用独立的矿井/客户编码体系，无主外键关联，跨系统分析需要通过主数据映射表实现。同一矿井在不同系统中的名称存在细微差异。

### 核心数据问题一览

| 问题 | 描述 |
|--------|------|
| 主数据不一致 | SAP 中客户编码6位，PI 中矿井编码无统一格式，LIMS 中矿井编码3位 |
| 时序数据质量问题 | 约0.5%点位缺失（设备掉线），约1%异常突升（传感器故障） |
| 订单-发货不一致 | VBAP 约1%关联到无效订单号 |
| 煤质与订单脱节 | LIMS 批次号与 SAP 物料批次无直接关联 |
| OA 流程数据资产化不足 | 合同编号规则不统一，无法按时间序列分析 |

---

## 4. 数据规模与质量问题注入

本项目生成了约 **1GB** 的模拟数据，覆盖 **2022-01 至 2023-06**（18个月），总记录数约 **1亿行**。所有数据集均按真实场景比例注入了数据质量问题。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 辅助：显示大数据框的简化版
def head(df, n=5):
    """安全地显示 DataFrame 前n行"""
    with pd.option_context('display.max_columns', 10, 'display.width', 200):
        return df.head(n)


import os
DATA_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'data', 'historical')
print(f"数据路径: {DATA_ROOT}")

### 4.1 数据规模总览

In [ ]:
import json

# 读取元数据
meta = json.load(open(os.path.join(DATA_ROOT, 'metadata.json')))
print("=== 模拟数据生成元数据 ===")
print(f"生成时间: {meta['generated_at']}")
print(f"耗时: {meta['duration_seconds']:.1f} 秒")
print(f"总大小: {meta['total_size_mb']} MB")
print()

# 数据规模表格
scale_info = [
    {"系统", "表/数据集", "记录数", "说明"},
    {"SAP-ERP", "KNA1 (客户主数据)", "15,000", "客户编码、名称、信用代码"},
    {"SAP-ERP", "VBAK (销售订单抬头)", "6,030,000", "~600万订单×2年"},
    {"SAP-ERP", "VBAP (销售订单行项目)", "12,060,000", "每订单平均2行"},
    {"PI-System", "tags (时序数据)", "78,624,000", "100标签×1min间隔×18月"},
    {"LIMS", "samples (煤质检测)", "2,010,000", "各矿井批次型煤质数据"},
    {"OA", "doc_flow (流程记录)", "5,025,000", "审批、请款等各类流程"},
    {"合计", "", "~1亿", "~995.6 MB"},
]

df_scale = pd.DataFrame(scale_info[1:], columns=scale_info[0])
print(df_scale.to_string(index=False))

### 4.2 质量问题注入比例

各数据集均按以下比例注入了模拟的数据质量问题，模拟真实生产环境中的数据缺陷：

In [ ]:
# 质量问题注入比例
quality_rules = pd.DataFrame({
    '系统': ['SAP-ERP', 'SAP-ERP', 'SAP-ERP', 'SAP-ERP', 'PI-System', 'PI-System', 'LIMS', 'OA'],
    '质量问题类型': ['空值 (null)', '异常值 (outlier)', '重复行 (duplicate)', '关联失效 (VBELN=0000)', '点位缺失', '异常突升', '空值/重复', '空值/重复'],
    '注入比例': ['0.5%', '0.5%', '0.5%', '~1%', '0.5%', '1.0%', '~1.5%', '~1.5%'],
})
print(quality_rules.to_string(index=False))

---

## 5. 各系统模拟数据预览

In [ ]:
# ── SAP-ERP ──
print("=" * 60)
print("【SAP-ERP】客户主数据 KNA1")
print("=" * 60)
kna1 = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'kna1.parquet'))
print(f"记录数: {len(kna1):,} | 列: {list(kna1.columns)}")
print(kna1.head(3).to_string(index=False))

print()
print("=" * 60)
print("【SAP-ERP】销售订单 VBAK (2023年样本)")
print("=" * 60)
vbak = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbak_year=2023.parquet'))
print(f"记录数: {len(vbak):,} | 列: {list(vbak.columns)}")
print(vbak.head(3).to_string(index=False))

print()
print("=" * 60)
print("【SAP-ERP】销售订单行项目 VBAP (2023年样本)")
print("=" * 60)
vbap = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbap_year=2023.parquet'))
print(f"记录数: {len(vbap):,} | 列: {list(vbap.columns)}")
print(vbap.head(3).to_string(index=False))

In [ ]:
# ── PI-System ──
print("=" * 60)
print("【PI-System】时序数据 (2022年1月样本)")
print("=" * 60)
pi = pd.read_parquet(os.path.join(DATA_ROOT, 'pi_system', 'tags_year=2022_month=01.parquet'))
print(f"记录数: {len(pi):,} | 列: {list(pi.columns)}")
print(f"标签数量: {pi['tag'].nunique()}")
print()
print("标签列表 (前20个):")
print(pi['tag'].unique()[:20])
print()
print("按传感器类型统计:")
pi['sensor_type'] = pi['tag'].str.extract(r'(WAGAS|TEMP|CO|CO2|PRESS|FAN_SPEED)')
print(pi['sensor_type'].value_counts())
print()
print("数据预览:")
print(pi.head(6).to_string(index=False))

In [ ]:
# ── LIMS ──
print("=" * 60)
print("【LIMS】煤质检测数据 (2023年样本)")
print("=" * 60)
lims = pd.read_parquet(os.path.join(DATA_ROOT, 'lims', 'samples_year=2023.parquet'))
print(f"记录数: {len(lims):,} | 列: {list(lims.columns)}")
print()
print("煤种分布:")
print(lims['SAMPLE_TYPE'].value_counts())
print()
print("矿井分布:")
print(lims['MINE_CODE'].value_counts())
print()
print("数据预览:")
print(lims.head(3).to_string(index=False))

In [ ]:
# ── OA ──
print("=" * 60)
print("【OA】流程审批数据 (2022年样本)")
print("=" * 60)
oa = pd.read_parquet(os.path.join(DATA_ROOT, 'oa', 'doc_flow_year=2022.parquet'))
print(f"记录数: {len(oa):,} | 列: {list(oa.columns)}")
print()
print("流程类型分布:")
print(oa['FLOW_TYPE'].value_counts())
print()
print("流程状态分布:")
print(oa['STATUS'].value_counts())
print()
print("数据预览:")
print(oa.head(3).to_string(index=False))

---

## 6. 矿井编码不一致问题（数据孤岛示例）

同一座矿井在不同系统中的编码格式不同，这是数据孤岛的核心表现：

In [ ]:
# 矿井编码不一致示例
print("=== 矿井编码不一致问题 ===\n")

print("【SAP-ERP VBAK】矿井工厂字段 WERKS 唯一值:")
print(vbak['WERKS'].unique())

print("\n【PI-System】矿井字段 mine 唯一值:")
print(pi['mine'].unique())

print("\n【LIMS】矿井编码字段 MINE_CODE 唯一值:")
print(lims['MINE_CODE'].unique())

print("\n【问题说明】")
print("各系统矿井编码格式不同：")
print("  SAP: WERKS = CN01/CN02/...  →  需映射到 M001/M002/...")
print("  PI:  mine = M001/M002/...     →  标准4位格式")
print("  LIMS: MINE_CODE = M001-M005  →  标准4位格式")
print("  OA:  无矿井字段              →  需通过合同关联")
print("\n这导致跨系统产销分析无法自动关联，需建立主数据映射表。")

---

## 7. 模拟数据生成方法

数据由项目中的 Python 生成器基于向量化方式批量生成：

```bash
# 生成历史数据（~1GB，约3分钟）
uv run python scripts/generate_historical.py

# 生成每日增量
uv run python scripts/generate_incremental.py 2024-01-01 2024-01-31

# 生成 SCADA 实时流（每秒推送Kafka消息）
uv run python -m dg_simulator.scada_simulator
```

生成器采用全向量化实现（NumPy/Pandas），100标签 × 1分钟间隔 × 18个月 ≈ 7862万条时序记录在约2分钟内生成完毕。

---

## 8. 数据治理四阶段路径

基于以上数据现状，A公司制定了分阶段推进的数据治理路线：

| 阶段 | 目标 | 说明 |
|--------|------|------|
| **Phase 1** | ODS贴源层建设 | 数据采集、CDC入湖、历史数据入湖 |
| **Phase 2** | 主数据管理（MDM） | 矿井、客户、物料编码标准化 |
| **Phase 3** | DWD主题层 + 质量监控 | 清洗规则、质量监控、问题工单 |
| **Phase 4** | 数据资产目录 + 血缘 | 元数据采集、血缘追踪、资产地图 |
| **Phase 5** | DWA应用层 + OLAP | 产销分析、安全预警、即席查询 |

**本 Demo 将依次展示 Phase 1 的核心能力。**